In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [9]:
df = pd.read_pickle("processed_reviews.pkl")

print(f"shape: {df.shape}")
print(df.info())

shape: (169612, 18)
<class 'pandas.DataFrame'>
RangeIndex: 169612 entries, 0 to 169611
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   review_id          169612 non-null  int64  
 1   source             169612 non-null  str    
 2   country            169552 non-null  str    
 3   province           169552 non-null  str    
 4   region_1           141654 non-null  str    
 5   region_2           66852 non-null   str    
 6   winery             169612 non-null  str    
 7   variety            169612 non-null  str    
 8   designation        119494 non-null  str    
 9   points             169612 non-null  int64  
 10  price              156796 non-null  float64
 11  description        169612 non-null  str    
 12  style_group        169612 non-null  str    
 13  batch              169612 non-null  int64  
 14  tokens_raw         169612 non-null  object 
 15  tokens             169612 non-null  object

# Preparing the data or modeling 
**STEPS**

1. Preprocess the data by dropping columns that are not needed
2. Create a train split with training, validation, and test dataset.  (70% train, 15% val, 15% test) 
3. apply TF-IDF onto the tokenized descriptions
4. Train the model and test 

In [10]:
y = df["style_group"]
print(df.shape, y.nunique(), "style groups")
y.value_counts()

(169612, 18) 31 style groups


style_group
Pinot Noir                                   16943
Chardonnay                                   15647
Cabernet Sauvignon                           13290
Bordeaux-style Red Blend                     13243
Red Blend (generic)                          12206
Syrah                                         7548
Sauvignon Blanc                               7290
Riesling                                      6663
Sparkling                                     5771
Rhône & Grenache Reds                         5579
Italian Reds                                  4875
Sangiovese                                    4799
Merlot                                        4757
Iberian Reds                                  4506
Rosé                                          4109
Zinfandel                                     4099
Pinot Gris & Pinot Blanc                      4028
Malbec                                        3446
Chenin, Semillon & Loire/SW France Whites     3325
White Blend (generi

**NOTE**:
The style groups are a bit imbalanced.... This may draw for some concern of possible underfitting of certain groups; particularly style groups bellow 1000 reviews. 

# Train / Validation/ Test ..... split (70, 15, 15)

In [11]:
train_val, test = train_test_split(df, test_size=0.15, stratify=y, random_state=67)

train, val = train_test_split(train_val, test_size=0.15 / 0.85, stratify=train_val["style_group"], random_state=67)

y_train, y_val, y_test = train["style_group"], val["style_group"], test["style_group"]
print(f"train {len(train):,} | val {len(val):,} | test {len(test):,}")

train 118,728 | val 25,442 | test 25,442


In [12]:
def make_preprocessor(text=False, desig=False, cat_cols=(), num_cols=()):
    steps = []
    if text:
        steps.append(("text", TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True), "clean_text")) # runs the text through TF-IDF if combinations requires this to be made
    if desig:
        steps.append(("desig", TfidfVectorizer(min_df=5), "clean_designation")) # Tokenized section 
    if cat_cols:
        steps.append(("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=5), #one hot encodes
                      list(cat_cols)))
    if num_cols:
        steps.append(("num", Pipeline([("impute", SimpleImputer(strategy="median")), # imputes median to nan values 
                                       ("scale", StandardScaler())]), # normalizes every value to maintain the same scale
                      list(num_cols)))
    return ColumnTransformer(steps) # this will piepline everything or transform the data if needed 


In [13]:
results = []

def run(name, preprocessor):
    models ={
        "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "LinearSVM": LinearSVC(class_weight="balanced")
    }
    fitted = {}
    for model_name, model in models.items():
        pipe = Pipeline([("prep", preprocessor), ("clf", model)])
        pipe.fit(train, y_train)
        pred = pipe.predict(val)
        results.append({"features": name, "model": model_name,
                        "val_accuracy": accuracy_score(y_val, pred),
                        "val_macro_f1": f1_score(y_val, pred, average="macro")})
        fitted[model_name] = pipe
        print(f"{name:} {model_name:} macro-F1 {results[-1]['val_macro_f1']:.3}")
    return fitted

# Test Combination....Baseline model

In [14]:
A = run("A: text", make_preprocessor(text=True))
B = run("B: location + numeric", make_preprocessor(cat_cols=["country", "province", "region_1", "region_2"], num_cols=["points", "price"]))
C = run("C: text + location", make_preprocessor(text=True, cat_cols=["country", "province", "region_1", "region_2"]))
D = run("D: text + location + numeric", make_preprocessor(text=True, cat_cols=["country", "province", "region_1", "region_2"], num_cols=["points", "price"]))
E = run("E: D + designation", make_preprocessor(text=True, desig=True, cat_cols=["country", "province", "region_1", "region_2"], num_cols=["points", "price"]))
F = run("F: E + winery", make_preprocessor(text=True, desig=True, cat_cols=["country", "province", "region_1", "region_2"] + ["winery"], num_cols=["points", "price"]))

pd.DataFrame(results).sort_values("val_macro_f1", ascending=False).round(3)

A: text LogReg macro-F1 0.476
A: text LinearSVM macro-F1 0.461
B: location + numeric LogReg macro-F1 0.492
B: location + numeric LinearSVM macro-F1 0.494
C: text + location LogReg macro-F1 0.673
C: text + location LinearSVM macro-F1 0.674
D: text + location + numeric LogReg macro-F1 0.676
D: text + location + numeric LinearSVM macro-F1 0.676
E: D + designation LogReg macro-F1 0.697
E: D + designation LinearSVM macro-F1 0.704
F: E + winery LogReg macro-F1 0.715
F: E + winery LinearSVM macro-F1 0.718


,features,model,val_accuracy,val_macro_f1
11,F: E + winery,LinearSVM,0.734,0.718
10,F: E + winery,LogReg,0.726,0.715
9,E: D + designation,LinearSVM,0.720,0.704
8,E: D + designation,LogReg,0.708,0.697
6,D: text + location + numeric,LogReg,0.688,0.676
7,D: text + location + numeric,LinearSVM,0.693,0.676
5,C: text + location,LinearSVM,0.691,0.674
4,C: text + location,LogReg,0.685,0.673
3,B: location + numeric,LinearSVM,0.488,0.494
2,B: location + numeric,LogReg,0.483,0.492


# Note
This is our baseline models... 

the model performed quite well overall. Particularly well with the teext data. As we can see the data has a decently high macro F1 